In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.agents import AgentState

class CustomState(AgentState):
    fav_colour: str

## Write to state

In [3]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def update_fav_colour(fav_colour: str, runtime: ToolRuntime) -> Command:
    """Update the favourite color of the user in the state once they've revealed it."""
    return Command(update={
        "fav_colour": fav_colour,
        "messages": [ToolMessage("Successfully updated favourite colour", tool_call_id=runtime.tool_call_id)]
    })

In [4]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    "gpt-5-nano",
    tools=[update_fav_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [5]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

In [6]:
from pprint import pprint

pprint(response)

{'fav_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='82e59075-36fe-4f64-9f40-f021a18feadc'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 349, 'prompt_tokens': 141, 'total_tokens': 490, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 320, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-E6L0tpU9xKOcXCN16Qw2y9PBsdAhj', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fa4fc-a944-7111-8b49-25bd84937732-0', tool_calls=[{'name': 'update_fav_colour', 'args': {'fav_colour': 'green'}, 'id': 'call_774FSRmyLfmnmgNTkw95OW4f', 'type': 'tool_call'}], invalid_tool_calls=[], usa

In [7]:
response = agent.invoke(
    {
        "messages": [HumanMessage(content="Hello, how are you?")],
        "fav_color": "green"
    },
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

{'messages': [HumanMessage(content='Hello, how are you?', additional_kwargs={}, response_metadata={}, id='c0b60e79-4427-41ed-aad9-bf0492c6bdb6'),
              AIMessage(content='Hi! I’m here and ready to help. How can I assist you today? If you’d like, we can chat about a topic, answer a question, work on a project, or tackle a task you have in mind.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 504, 'prompt_tokens': 142, 'total_tokens': 646, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 448, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-E6L2i1YiiqdpNKEYTjFnMdxXRIBR9', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019fa4fe-64c4-7931-a02c-54f616d0ad9e-0', tool_calls=[], invalid_tool_

## Read state

In [8]:
@tool
def read_fav_colour(runtime: ToolRuntime) -> str:
    """Read the favourite color of the user from the state."""
    try:
        return runtime.state["fav_colour"]
    except KeyError:
        return "No favourite color found in state"
    
agent = create_agent(
    "gpt-5-nano",
    tools=[update_fav_colour, read_fav_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)
    

In [9]:
response = agent.invoke(
    {"messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'fav_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='2487110b-553d-4caa-8e52-3b82c7dd61a6'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 218, 'prompt_tokens': 162, 'total_tokens': 380, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 192, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-E6L6wjwTaqoNxocM2BDY6bGD4LAvQ', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fa502-6357-7113-a68d-23355466950f-0', tool_calls=[{'name': 'update_fav_colour', 'args': {'fav_colour': 'green'}, 'id': 'call_KbvWJRnzN1EhYHaZyd3b6E0g', 'type': 'tool_call'}], invalid_tool_calls=[], usa

In [10]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What's my favourite colour?")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'fav_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='2487110b-553d-4caa-8e52-3b82c7dd61a6'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 218, 'prompt_tokens': 162, 'total_tokens': 380, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 192, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-E6L6wjwTaqoNxocM2BDY6bGD4LAvQ', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fa502-6357-7113-a68d-23355466950f-0', tool_calls=[{'name': 'update_fav_colour', 'args': {'fav_colour': 'green'}, 'id': 'call_KbvWJRnzN1EhYHaZyd3b6E0g', 'type': 'tool_call'}], invalid_tool_calls=[], usa